# TP 1 — Installer son atelier data

**Durée : 2 h.** Vous connaissez SQL. Le travail consiste à préparer un environnement dans lequel vous pourrez collecter, traiter et livrer des données.

Suivre `TP1_ENVIRONNEMENT.md`. À la fin, le notebook doit fonctionner dans VS Code avec un noyau Colab. Les trois premières cellules sont fournies ; les trois requêtes finales sont à écrire.

Les données décrivent plusieurs réseaux électriques américains sur un semestre. Aucune connaissance préalable du secteur n'est nécessaire.

## 1. Vérifier où le code s'exécute
Exécuter cette cellule. Expliquer la différence entre le dossier Git sur votre ordinateur et le dossier affiché par le noyau.

In [ ]:
from pathlib import Path
import sys
print("Python :", sys.version.split()[0])
print("Dossier :", Path.cwd())
print("Dossier Colab présent :", Path("/content").exists())

## 2. Installer DuckDB et activer les cellules SQL
Exécuter les deux cellules fournies. Ne pas ajouter JupySQL ni `%load_ext sql`. La fonction fournie transmet simplement les cellules `%%sql` à DuckDB ; elle ne fait aucune analyse des données.

In [ ]:
%pip install -q "duckdb==1.4.3"

In [ ]:
from pathlib import Path
import duckdb
from IPython.core.magic import register_cell_magic
from IPython.display import display

# Chemins relatifs : les fichiers sont sur la machine qui exécute le noyau.
Path("data").mkdir(exist_ok=True)
Path("exports").mkdir(exist_ok=True)
if "db" in globals():
    db.close()
db = duckdb.connect("data/atelier.duckdb")

# Petit utilitaire fourni : il permet d'écrire des cellules SQL.
@register_cell_magic
def sql(line, cell):
    if cell.strip() and any(not x.strip().startswith("--") and x.strip() for x in cell.splitlines()):
        result = db.execute(cell)
        if result.description:
            display(result.fetch_df())

print("Dossier d'exécution :", Path.cwd())

In [ ]:
%%sql
SELECT 'Connexion SQL prête' AS message, 2 + 3 AS resultat;

## 3. Récupérer les données
Exécuter le téléchargement fourni, puis vérifier le nom et la taille du fichier. Le fichier fait environ 42 Mo. Ne pas le committer dans Git.

Le notebook conserve toutes les colonnes sous forme de texte pour commencer. Le typage fera partie du pipeline au TP 2.

In [ ]:
from urllib.request import urlretrieve

url = "https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/EIA930_BALANCE_2024_Jan_Jun.csv"
fichier = Path("data/eia930_2024_s1.csv")

if not fichier.exists():
    temporaire = fichier.with_suffix(".part")
    urlretrieve(url, temporaire)
    temporaire.replace(fichier)

print("Fichier prêt :", fichier, "—", round(fichier.stat().st_size / 1_000_000, 1), "Mo")

In [ ]:
%%sql
CREATE OR REPLACE VIEW brut AS
SELECT *
FROM read_csv('data/eia930_2024_s1.csv', header = true, all_varchar = true);

SELECT * FROM brut LIMIT 5;

## 4. Trois requêtes à écrire

**A. Inventaire.** Combien de lignes et de réseaux distincts contient le fichier ? Le réseau est identifié par `"Balancing Authority"`.

Utiliser `LIMIT` pour les aperçus : ne pas afficher toute la source dans le notebook.

In [ ]:
%%sql
-- À vous de coder : compter les lignes et les réseaux distincts de brut.

**B. Répartition.** Compter les lignes par réseau et trier le résultat. Tous les réseaux ont-ils le même nombre de lignes ? Ne rien supprimer à ce stade.

In [ ]:
%%sql
-- À vous de coder : compter les lignes par "Balancing Authority" et trier.

**C. Première sélection.** Choisir un réseau, afficher dix lignes avec date, fin d'heure locale, demande et production totale. Renommer les colonnes pour rendre le tableau lisible.

Colonnes : `"Data Date"`, `"Local Time at End of Hour"`, `"Demand (MW)"`, `"Net Generation (MW)"`.

In [ ]:
%%sql
-- À vous de coder : sélectionner dix lignes du réseau de votre choix avec des alias lisibles.

## 5. Expliquer et sauvegarder

À compléter ci-dessous :
- Le code s'exécute sur…
- Le notebook est sauvegardé dans…
- Les données téléchargées sont dans…
- Une ligne représente…
- Mon réseau choisi est…

Enregistrer le notebook, puis faire le commit demandé dans le support TP 1. Être capable d'expliquer une requête et de changer son filtre en direct.